In [105]:
import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
import matplotlib.pyplot as plt
from pyampute.exploration.mcar_statistical_tests import MCARTest
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import r2_score

In [61]:
df = pd.read_csv("data/Life Expectancy Data.csv")
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_").str.replace('/', '_').str.replace('-', '_')
df.head()

,country,year,status,life_expectancy,adult_mortality,infant_deaths,alcohol,percentage_expenditure,hepatitis_b,measles,...,polio,total_expenditure,diphtheria,hiv_aids,gdp,population,thinness__1_19_years,thinness_5_9_years,income_composition_of_resources,schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [36]:
df.describe()

,year,life_expectancy,adult_mortality,infant_deaths,alcohol,percentage_expenditure,hepatitis_b,measles,bmi,under-five_deaths,polio,total_expenditure,diphtheria,hiv/aids,gdp,population,thinness__1-19_years,thinness_5-9_years,income_composition_of_resources,schooling
count,2938.000000,2928.000000,2928.000000,2938.000000,2744.000000,2938.000000,2385.000000,2938.000000,2904.000000,2938.000000,2919.000000,2712.00000,2919.000000,2938.000000,2490.000000,2.286000e+03,2904.000000,2904.000000,2771.000000,2775.000000
mean,2007.518720,69.224932,164.796448,30.303948,4.602861,738.251295,80.940461,2419.592240,38.321247,42.035739,82.550188,5.93819,82.324084,1.742103,7483.158469,1.275338e+07,4.839704,4.870317,0.627551,11.992793
std,4.613841,9.523867,124.292079,117.926501,4.052413,1987.914858,25.070016,11467.272489,20.044034,160.445548,23.428046,2.49832,23.716912,5.077785,14270.169342,6.101210e+07,4.420195,4.508882,0.210904,3.358920
min,2000.000000,36.300000,1.000000,0.000000,0.010000,0.000000,1.000000,0.000000,1.000000,0.000000,3.000000,0.37000,2.000000,0.100000,1.681350,3.400000e+01,0.100000,0.100000,0.000000,0.000000
25%,2004.000000,63.100000,74.000000,0.000000,0.877500,4.685343,77.000000,0.000000,19.300000,0.000000,78.000000,4.26000,78.000000,0.100000,463.935626,1.957932e+05,1.600000,1.500000,0.493000,10.100000
50%,2008.000000,72.100000,144.000000,3.000000,3.755000,64.912906,92.000000,17.000000,43.500000,4.000000,93.000000,5.75500,93.000000,0.100000,1766.947595,1.386542e+06,3.300000,3.300000,0.677000,12.300000
75%,2012.000000,75.700000,228.000000,22.000000,7.702500,441.534144,97.000000,360.250000,56.200000,28.000000,97.000000,7.49250,97.000000,0.800000,5910.806335,7.420359e+06,7.200000,7.200000,0.779000,14.300000
max,2015.000000,89.000000,723.000000,1800.000000,17.870000,19479.911610,99.000000,212183.000000,87.300000,2500.000000,99.000000,17.60000,99.000000,50.600000,119172.741800,1.293859e+09,27.700000,28.600000,0.948000,20.700000


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   country                          2938 non-null   str    
 1   year                             2938 non-null   int64  
 2   status                           2938 non-null   str    
 3   life_expectancy                  2928 non-null   float64
 4   adult_mortality                  2928 non-null   float64
 5   infant_deaths                    2938 non-null   int64  
 6   alcohol                          2744 non-null   float64
 7   percentage_expenditure           2938 non-null   float64
 8   hepatitis_b                      2385 non-null   float64
 9   measles                          2938 non-null   int64  
 10  bmi                              2904 non-null   float64
 11  under-five_deaths                2938 non-null   int64  
 12  polio                          

In [48]:
(df.isnull().mean()*100).sort_values(ascending=False) 

population                         22.191967
hepatitis_b                        18.822328
gdp                                15.248468
total_expenditure                   7.692308
alcohol                             6.603131
income_composition_of_resources     5.684139
schooling                           5.547992
thinness__1-19_years                1.157250
thinness_5-9_years                  1.157250
bmi                                 1.157250
diphtheria                          0.646698
polio                               0.646698
life_expectancy                     0.340368
adult_mortality                     0.340368
infant_deaths                       0.000000
status                              0.000000
country                             0.000000
year                                0.000000
under-five_deaths                   0.000000
measles                             0.000000
percentage_expenditure              0.000000
hiv/aids                            0.000000
dtype: flo

In [73]:
df['status'].value_counts()

status
Developing    2426
Developed      512
Name: count, dtype: int64

<h4>that means the data is heavily skewed towards the Developing Countries</h4>

It can be a problem if the training data has only few Developed Countries

---

<h4>Let's first Decide if the data is MCAR/MAR/MNAR </h4>

---

To decide that let's create a function that takes the df and returns the imputed df

In [ ]:
def check_missingness(df: pd.DataFrame):

    treated_df = df.copy()

    encoded_df = df.copy()


    # Temparory Encoding 
    label_encoders = {}
    for col in encoded_df.select_dtypes(include=['object']).columns.tolist():
        label_encoders[col] = LabelEncoder()
        
        encoded_df[col] = label_encoders[col].fit_transform(encoded_df[col])
    

    # Identify the Features with no Null values 
    missing_percentages = df.isnull().mean()

    complete_features = missing_percentages[missing_percentages == 0].index.tolist()

    # If No column are 100% complete pick columns with <1% missingness
    if len(complete_features) < 2:
        complete_features = missing_percentages[missing_percentages < 0.01].index.tolist()

        encoded_df = encoded_df.dropna(subset = complete_features)
    
    print("Running MCAR Test....")
    mcar_test = MCARTest()
    p_value = mcar_test(df.select_dtypes(exclude=['object']))
    print(f"p value : {p_value}")

    # If p_value > 0.05 means Null Hypothesis is Correct Data is MCAR
    if p_value > 0.05: 
        # Treat as MCAR
        print("Data is MCAR")
        num_cols = treated_df.select_dtypes(exclude=['object']).columns.tolist()

        for col in num_cols:
            if treated_df[col].isnull().any():
                treated_df[col] = treated_df[col].fillna(treated_df[col].median())
        return treated_df

    # If p_value < 0.05 
    print("Data is Not MCAR")
    print("Checking btw MAR and MNAR")

    columns_to_test = missing_percentages[missing_percentages > 0].index.tolist()
    mar_cols = []
    mnar_cols = []

    # Removing the country columns because lower gdp and population countries will never give there name so my model don't learn it memorize that if this country came the value will be missing 
    complete_features.remove('country')

    

    # Checking each row individually
    for col in columns_to_test:

        pct = missing_percentages[col]*100

        # if missing values are <1% then just fill it with mean or median
        if pct < 1:
            treated_df[col] = treated_df[col].fillna(treated_df[col].median())
            continue

        # creating a missing indicator for model
        encoded_df['is_missing'] = encoded_df[col].isnull().astype(int) 
        
        X = encoded_df[complete_features]
        y = encoded_df['is_missing']

        clf = RandomForestClassifier(n_estimators=100, random_state=0)

        scores = cross_val_score(clf, X, y, cv = 5, scoring='roc_auc')

        mean_auc = scores.mean()
        print(f"AUC for {col} : {mean_auc}")
        # If mean_auc >= 0.70 the columns has high relation and will be added to mar_cols
        if mean_auc >= 0.70:
            mar_cols.append(col)
        # if not then it is suspected to be MNAR and need to add missing indicator
        else:
            treated_df[f"{col}_was_missing"] = treated_df[col].isnull().astype(int)
            mnar_cols.append(col)
    print(f"MAR Columns : {mar_cols}")
    print(f"MNAR Columns : {mnar_cols}")
    # Imputation for MAR/MNAR columns
    if mar_cols or mnar_cols:
        num_cols = treated_df.select_dtypes(exclude='object').columns.tolist()

        mice = IterativeImputer(max_iter=50, random_state=0)

        treated_df[num_cols] = mice.fit_transform(treated_df[num_cols])
    print("Data Imputation is complete")
    return treated_df  

In [104]:
imputed_df = check_missingness(df)

C:\Users\Keval\AppData\Local\Temp\ipykernel_17688\1465827037.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in encoded_df.select_dtypes(include=['object']).columns.tolist():


Running MCAR Test....
p value : 0.0
Data is Not MCAR
Checking btw MAR and MNAR
AUC for alcohol : 0.9833339451809312
AUC for hepatitis_b : 0.7965645233569761
AUC for bmi : 0.7993462378719663
AUC for total_expenditure : 0.967533871135012
AUC for gdp : 0.990176235528882
AUC for population : 0.8519782532839745
AUC for thinness__1_19_years : 0.7993462378719663
AUC for thinness_5_9_years : 0.7993462378719663
AUC for income_composition_of_resources : 0.8549804218396275
AUC for schooling : 0.8607807807807808
MAR Columns : ['alcohol', 'hepatitis_b', 'bmi', 'total_expenditure', 'gdp', 'population', 'thinness__1_19_years', 'thinness_5_9_years', 'income_composition_of_resources', 'schooling']
MNAR Columns : []
Data Imputation is complete


In [102]:
imputed_df.skew(numeric_only=True)

year                               -0.006409
life_expectancy                    -0.642391
adult_mortality                     1.177899
infant_deaths                       9.786963
alcohol                             0.603329
percentage_expenditure              4.652051
hepatitis_b                        -1.632939
measles                             9.441332
bmi                                -0.199786
under_five_deaths                   9.495065
polio                              -2.108909
total_expenditure                   0.645587
diphtheria                         -2.083566
hiv_aids                            5.396112
gdp                                 3.523582
population                         17.899691
thinness__1_19_years                1.711266
thinness_5_9_years                  1.776239
income_composition_of_resources    -1.088048
schooling                          -0.573388
dtype: float64

In [ ]:
status_encoder_pipe = Pipeline(steps = [
    ('encode', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

imputer_pipe = Pipeline(steps=[
    ('imputer', FunctionTransformer(func=check_missingness))
])

scaler_pipe = Pipeline(steps = [
    ('scaler', StandardScaler())
])

skewness_pipe = Pipeline(steps = [
    ('yeo-johnson', PowerTransformer(method='yeo-johnson'))
])



In [ ]:
vif = pd.DataFrame()
vif['column'] = df.columns
vif['vif'] = [variance_inflation_factor(df.values,i) for i in range(df.shape[1])]
vif